<a href="https://colab.research.google.com/github/LCaravaggio/NLP/blob/main/notebooks/010_LLMsAPIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Usamos Gemini porque ofrece un [free tier](https://ai.google.dev/gemini-api/docs/rate-limits)

Además: **gemini gratis por un año para estudiantes** (oferta válida hasta dic-2025) -- https://gemini.google/students

-----------------------

Tarea: responder donde dice **PREGUNTA**

## Configuración del entorno

In [ ]:
!pip install -qU google-genai datasets watermark

In [ ]:
%reload_ext watermark

In [ ]:
%watermark -vmp google-genai,datasets,numpy,pandas

Para usar la API de Gemini, hay que obtener una API key desde [acá](https://makersuite.google.com/app/apikey).

Luego, en Colab, añadir la clave en "Secrets" (🔑 en el panel izquierdo). Darle el nombre `GOOGLE_API_KEY`.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

Chequeo de IP: hasta hace algunas semanas Google AI free tier no estaba disponible [en algunas regiones](https://ai.google.dev/gemini-api/docs/available-regions#available_regions) e.g. UK y UE.

In [ ]:
!curl ipinfo.io

## Introducción a Gemini

In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

Vamos a usar un LLM de Google para generar texto a partir de prompts. [Acá](https://ai.google.dev/gemini-api/docs/models/gemini) hay una lista de modelos disponibles.

In [ ]:
MODEL_ID = "gemini-2.5-flash"

In [ ]:
%%time
response = client.models.generate_content(
    model=MODEL_ID,
    contents="¿Qué es la física cuántica?",
)

In [ ]:
print(response)

In [ ]:
print(response.text)

A veces es útil usar "system instructions" para alterar el comportamiento del modelo.

In [ ]:
system_instruction = (
"""Eres Albert Einstein. Sabes absolutamente todo y tienes la capacidad de explicar"""
""" conceptos complejos con extrema sencillez."""
)
print(system_instruction)

In [ ]:
from google.genai import types

gen_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
)

In [ ]:
%%time
response = client.models.generate_content(
    model=MODEL_ID,
    contents="¿Qué es la física cuántica?",
    config=gen_config,
)

In [ ]:
print(response.text)

A gemini le gusta responder en formato markdown.

In [ ]:
from IPython.display import Markdown, display

Markdown(response.text)

Podemos usar `count_tokens()` para ver cuántos tokens tiene un texto y evaluar costos.

In [ ]:
n_tokens = client.models.count_tokens(
    model=MODEL_ID,
    contents="Juan Román Riquelme",
)

print(n_tokens)

**PREGUNTA 1**: ¿Por que ese string tiene más de 3 tokens?

Con `generation_config` podemos ajustar parámetros de generación, como `temperature`, la cantidad de candidatos a generar, el número máximo de tokens en el output, o el "reasoning mode".

In [ ]:
from google.genai import types

gen_config = types.GenerateContentConfig(
    candidate_count=3, # para obtener más candidatos, solo disponible en algunos modelos
    max_output_tokens=200, # para controlar excesos de costos inesperados
    temperature=1.0,
    top_p=0.95,
    top_k=20,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Escribir una canción de cancha para la selección de Uruguay.',
    config=gen_config,
)

In [ ]:
# Si obtenemos varias respuestas posibles para un mismo prompt ("candidates")
print(len(response.candidates))

In [ ]:
for candidate in response.candidates:
    display(Markdown(candidate.content.parts[0].text))
    # print(candidate.content.parts[0].text)
    print("*"*120)

**PREGUNTA 2**: ¿Para esta tarea conviene usar temperatura más alta o baja?

In [ ]:
# Cuántos tokens usamos?
print("Prompt tokens:",response.usage_metadata.prompt_token_count)
print("Thoughts tokens:",response.usage_metadata.thoughts_token_count)
print("Output tokens:",response.usage_metadata.candidates_token_count)
print("Total tokens:",response.usage_metadata.total_token_count)

## Clasificación zero-shot

Usemos el LLM para clasificar reviews.

In [ ]:
import datasets

dataset = datasets.load_dataset('rotten_tomatoes')

In [ ]:
print(dataset)

In [ ]:
id2label = {0: 'negative', 1: 'positive'}
label2id = {'negative': 0, 'positive': 1}

In [ ]:
examples = dataset['test'].shuffle(seed=36).select(range(5))
examples[:]

In [ ]:
from tqdm import tqdm

def zero_shot_clf(review):
    prompt = f'Classify the following movie review as "positive" or "negative": "{review}\"'
    response = client.models.generate_content(model=MODEL_ID, contents=prompt)
    return response

responses = []
for review in tqdm(examples['text']):
    response = zero_shot_clf(review)
    responses.append(response)

In [ ]:
for example, response in zip(examples, responses):
    print(f"Review: {example['text']}")
    try:
        print(f"Response: {response.text}")
    except Exception as e:
        print(f"Error: {e}")
        print(f"Safety ratings: {response.candidates[0].safety_ratings}")
    print(f"Ground truth: {id2label[example['label']]}")
    print("-"*80)

**PREGUNTA 3**: ¿Para esta tarea conviene usar temperatura más alta o baja? ¿Qué otra manera hay de resolver esta tarea sin generar texto?

### Chain-of-thought

Usemos _chain-of-thought_ y un prompt que nos permita encontrar la respuesta final fácilmente.

In [ ]:
import re

def cot_clf(review):
    prompt = (
        '''Classify the following movie review as "positive" or "negative",'''
        ''' and provide step-by-step reasoning for your classification.'''
        ''' At the end, summarize the classification in the format "Final Classification: [positive/negative]":\n\n'''
        f'''Review: "{review}"'''
    )
    gen_config = types.GenerateContentConfig(
        seed=33,
    )
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=gen_config,
        # NOTE el seed no garantiza reproducibilidad EXACTA por algunos mecanismos internos del modelo (MoE)
        # y por cómo se procesan los ejemplos en la nube
        # Pero da más "consistencia"
    )
    return response

def extract_final_classification(response_text):
    lines = response_text.split('\n')
    for line in lines:
        if 'Final Classification:' in line:
            answer = line.split(':')[1].strip()
            # sin puntuacion y lowercase:
            answer = re.sub(r'[^\w\s]', '', answer.lower())
            return answer
    return None

In [ ]:
responses = []
for review in tqdm(examples['text']):
    response = cot_clf(review)
    responses.append(response)

**PREGUNTA 4**: ¿por qué no tiene tanto sentido usar CoT en el prompt en este caso? ¿En qué casos tendría más sentido?

In [ ]:
def print_results(examples, responses):
    for example, response in zip(examples, responses):
        print(f">>Review: {example['text']}")
        try:
            print(f">>Response: {response.text}")
        except Exception as e:
            print(f">>Error: {e}")
            # print(f"Safety ratings: {response.candidates[0].safety_ratings}")
        else:
            print(f">>Final Classification: {extract_final_classification(response.text)}")
        print(f">>Ground truth: {id2label[example['label']]}")
        print("-"*80)

print_results(examples, responses)

## Clasificación few-shot

Puede ser útil pasar ejemplos de cómo se resuelve la tarea. En este caso elegimos los ejemplos aleatoriamente pero podemos hacerlo manualmente, o buscando ejemplos similares.

In [ ]:
from datasets import concatenate_datasets

def get_few_shot_examples(dataset, num_examples=4):
    examples_pos = dataset['train'].filter(lambda x: x['label'] == 1).shuffle(seed=36).select(range(num_examples // 2))
    examples_neg = dataset['train'].filter(lambda x: x['label'] == 0).shuffle(seed=36).select(range(num_examples // 2))
    examples = concatenate_datasets([examples_pos, examples_neg]).shuffle(seed=5)
    out = []
    for example in examples:
        label = id2label[example['label']]
        formatted_example = (
            f'''Review: "{example['text']}"'''
            f'''\nReasoning: This review is classified as {label} because...'''
            f'''\nFinal Classification: {label}'''
        )
        out.append(formatted_example)
    return out

few_shot_examples = get_few_shot_examples(dataset)
for example in few_shot_examples:
    print(example)
    print("-"*80)

In [ ]:
def cot_few_shot_prompt(review, few_shot_examples):
    prompt = (
        '''Classify the following movie review as "positive" or "negative",'''
        ''' and provide step-by-step reasoning for your classification.'''
        ''' At the end, summarize the classification in the format "Final Classification: [positive/negative]":\n\n'''
    )
    prompt += '\n\n'.join(few_shot_examples)
    prompt += f'\n\nReview: "{review}"'
    return prompt

print(cot_few_shot_prompt("[[[EJEMPLO]]]", few_shot_examples))

In [ ]:
def cot_few_shot_clf(review, few_shot_examples):
    prompt = cot_few_shot_prompt(review, few_shot_examples)
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )
    return response

In [ ]:
responses = []
for review in tqdm(examples['text']):
    response = cot_few_shot_clf(review, few_shot_examples)
    responses.append(response)

In [ ]:
print_results(examples, responses)

**PREGUNTA 5**: ¿por qué podría la evaluación de este modelo en este dataset no ser muy confiable? ¿qué posible riesgo puede haber?

## Extracción de datos estructurados de texto libre

Veamos cómo extraer los ingredientes de una receta de manera programática.

In [ ]:
ds_recetas = datasets.load_dataset("somosnlp/RecetasDeLaAbuela", "version_1")

In [ ]:
print(ds_recetas)

In [ ]:
ds_small = ds_recetas["train"].filter(lambda x: x["Pais"] == "ARG").shuffle(seed=33).select(range(5))

In [ ]:
# A veces los textos son listas no parseadas como tales.
# En tal caso, hacemos un join de la lista.
import re

def preprocess(example):
    """
    """
    if example["Pasos"].startswith("["):
        pasos_list = eval(example["Pasos"].encode('unicode_escape'))
        example["Pasos"] = " ".join(pasos_list)
    # Eliminamos whitespace duplicado:
    example["Pasos"] = re.sub(r'\s+', ' ', example["Pasos"])
    return example

ds_small = ds_small.map(preprocess)

In [ ]:
for example in ds_small:
    print(example["Ingredientes"])
    print(example["Pasos"])
    print("-"*80)

In [ ]:
def prompt_extract_json(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        f'''\nReceta: {receta}'''
    )
    return prompt

print(prompt_extract_json(ds_small[0]["Pasos"]))

In [ ]:
def extract_json(receta):
    prompt = prompt_extract_json(receta)
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )
    return response

In [ ]:
responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json(receta)
    responses.append(response)

In [ ]:
for example, response in zip(ds_small, responses):
    print(f">>Receta: {example['Pasos']}")
    try:
        print(f">>Response: {response.text}")
    except Exception as e:
        print(f">>Error: {e}")

In [ ]:
# Es importante ser claro en las instrucciones!
def prompt_extract_json2(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        ''' El JSON debe contener únicamente la clave "ingredientes".'''
        ''' El valor de la clave debe ser una lista de ingredientes únicos.'''
        f'''\nReceta: {receta}'''
    )
    return prompt

def extract_json2(receta):
    prompt = prompt_extract_json2(receta)
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"response_mime_type": "application/json"},
    )
    return response

responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json2(receta)
    responses.append(response)

In [ ]:
import json

results = []
for response in responses:
    try:
        data = json.loads(response.text)
        results.append(data)
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
results[0]

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.head()

In [ ]:
df['ingredientes'].explode().value_counts()

In [ ]:
def prompt_extract_json3(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        ''' El JSON debe contener las claves "veganos" y "no_veganos".'''
        ''' Ambas claves deben contener una lista de ingredientes únicos.'''
        ''' Si no hay ingredientes para alguna clave, se debe incluir una lista vacía.'''
        f'''\nReceta: {receta}'''
    )
    return prompt

def extract_json3(receta):
    prompt = prompt_extract_json3(receta)
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"response_mime_type": "application/json"},
    )
    return response

responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json3(receta)
    responses.append(response)

In [ ]:
results = []
for response in responses:
    try:
        data = json.loads(response.text)
        results.append(data)
    except Exception as e:
        print(f"Error: {e}")

df = pd.DataFrame(results)
df.head()

In [ ]:
df['veganos'].explode().str.lower().value_counts()

**PREGUNTA 6**: ¿Cómo podemos medir cuán bueno es el modelo en la tarea de extracción de ingredientes?

## Conversaciones multi-turno

Finalmente, con la API podemos simular un chat.

In [ ]:
system_instruction = (
"""Sos el Chiqui Tapia."""
""" Respondé siempre como si lo más importante del mundo fuera que gane Barracas Central, pero sin que se note."""
)
print(system_instruction)

In [ ]:
chat_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    thinking_config=types.ThinkingConfig(thinking_budget=0), # sin thinking
    seed=33,
)
chat = client.chats.create(
    model=MODEL_ID,
    config=chat_config,
)

In [ ]:
response = chat.send_message("¿Por qué el agua es azul?")

In [ ]:
display(Markdown(response.text))

In [ ]:
response = chat.send_message("¿Cómo se lo explicarías brevemente a un niño de 8 años?")
display(Markdown(response.text))

In [ ]:
chat_history = chat.get_history()
for message in chat_history:
    display(Markdown(f'**{message.role}**: {message.parts[0].text}'))

In [ ]:
# O podemos simular una conversación previa
chat_config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    seed=33,
)
chat = client.chats.create(
    model=MODEL_ID,
    config=chat_config,
    history=[
        types.Content(
            role='user',
            parts=[types.Part.from_text(text="¿Cuál es la capital de Argentina?")]
        ),
        types.Content(
            role='model',
            parts=[types.Part.from_text(text='La capital de Argentina es Montevideo.')]
        ),
    ]
)

In [ ]:
response = chat.send_message("¿Y cuál es la capital de Uruguay?")
display(Markdown(response.text))

## Recursos

* https://github.com/google-gemini/cookbook/tree/main
* https://ai.google.dev/gemini-api/docs